In [ ]:
# Installing dependencies
!pip install gradio sounddevice transformers torchaudio librosa

In [ ]:
# Importing necessary libraries
import torch
import gradio as gr
import sounddevice as sd
import numpy as np
import librosa

from transformers import WhisperProcessor, WhisperForConditionalGeneration

In [ ]:
# Loading fine-tune model
processor = WhisperProcessor.from_pretrained("stt_model")
model = WhisperForConditionalGeneration.from_pretrained("stt_model")

In [ ]:
# Transcription Function
def transcribe(audio):
    audio = audio[1]
    audio = librosa.resample(audio.astype(np.float32), orig_sr=48000, target_sr=16000)

    inputs = processor(audio, sampling_rate=16000, return_tensors="pt").input_features
    predicted_ids = model.generate(inputs)
    transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]

    return transcription

In [ ]:
# Gradio App
interface = gr.Interface(
    fn=transcribe,
    inputs=gr.Audio(source="microphone", type="numpy"),
    outputs="text",
    title="Speech to Text App",
    description="Fine-Tuned Whisper STT Model"
)

interface.launch()

In [ ]:
# Quantization for faster inference
quantized_model = torch.quantization.quantize_dynamic(
    model, {torch.nn.Linear}, dtype=torch.qint8
)